# SL-7 Cross-Model Run — Qwen3-VL-2B-Instruct (family-gated cross-model protocol)

This notebook uses the **exact frozen SL-7 v16 images, prompts, answer keys, family confirmation rules, greedy decoding, and max_new_tokens=64**.

### Cross-model calibration amendment
The original single-model v16 protocol used one global calibration gate. During the cross-model extension, several otherwise capable models passed some prerequisite visual skills but failed an unrelated calibration family. Therefore, before inspecting any scored Qwen3-VL-2B holdout outputs, the cross-model protocol is amended to use **family-specific calibration eligibility**.

A failed calibration family does **not** validate the corresponding downstream blind-spot family; that family is reported as **INELIGIBLE / diagnostic only**. Unrelated families whose required calibration gates pass remain eligible. No images, prompts, gold answers, scored-family thresholds, or model-specific thresholds are changed.

Model: `Qwen/Qwen3-VL-2B-Instruct`.


In [1]:
PROTOCOL_VERSION = "SL7_CROSS_MODEL_FAMILY_GATED_V2"
print("="*72)
print(PROTOCOL_VERSION)
print("IMPORTANT: This notebook does NOT stop when an unrelated calibration family fails.")
print("It continues to the frozen holdout and marks affected blind-spot families INELIGIBLE.")
print("="*72)


SL7_CROSS_MODEL_FAMILY_GATED_V2
IMPORTANT: This notebook does NOT stop when an unrelated calibration family fails.
It continues to the frozen holdout and marks affected blind-spot families INELIGIBLE.


In [2]:
!pip install -q -U "transformers==5.17.0" "accelerate>=1.0" "huggingface_hub>=0.30"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 27.5 MB/s eta 0:00:00


In [3]:
import os, glob, csv, json, time, platform, sys, zipfile, importlib, random, gc
from pathlib import Path
from PIL import Image
import torch, transformers

MAX_NEW_TOKENS=64
INFERENCE_SEED=20261004
random.seed(INFERENCE_SEED); torch.manual_seed(INFERENCE_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(INFERENCE_SEED)

hits=[]
for p in glob.glob("/kaggle/input/**/dataset/items.csv", recursive=True):
    root=Path(p).parent.parent
    if (root/"VERSION_SL7_v16.txt").exists() and (root/"eval/score.py").exists():
        hits.append(root)
if not hits:
    raise FileNotFoundError("Add the snakes-ladders-SL7-multimodel-v1 package as a Kaggle input dataset.")
ROOT=hits[0]
ITEMS=list(csv.DictReader(open(ROOT/"dataset/items.csv",encoding="utf-8")))
CAL=[x for x in ITEMS if x["part"]=="CAL"]
SCORED=[x for x in ITEMS if x["part"]=="SCORED"]
print("Frozen project root:",ROOT)
print("Calibration:",len(CAL),"Scored:",len(SCORED))

sys.path.insert(0,str(ROOT/"eval"))
import score
importlib.reload(score)


Frozen project root: /kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1
Calibration: 20 Scored: 150


<module 'score' from '/kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1/eval/score.py'>

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText
MODEL_ID='Qwen/Qwen3-VL-2B-Instruct'
MODEL_SLUG='qwen3_vl_2b'
MODEL_DISPLAY='Qwen3-VL-2B-Instruct'
OUT=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}")
OUT.mkdir(exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator.")

# Optional HF login for gated checkpoints (especially Gemma).
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

load_kwargs=dict(device_map={"":0}, dtype=torch.float16, low_cpu_mem_usage=True)
try:
    model=AutoModelForImageTextToText.from_pretrained(MODEL_ID,attn_implementation="sdpa",trust_remote_code=False,**load_kwargs)
    attention="sdpa"
except Exception as e:
    print("SDPA load failed; retrying with model default attention:",repr(e))
    model=AutoModelForImageTextToText.from_pretrained(MODEL_ID,trust_remote_code=False,**load_kwargs)
    attention="model_default"
model.eval()
processor=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=False)

env={"model_key":'qwen3_vl_2b',"model_display":MODEL_DISPLAY,"model_id":MODEL_ID,"parameters":sum(p.numel() for p in model.parameters()),"transformers":transformers.__version__,"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0),"python":platform.python_version(),"dtype":"float16","attention":attention,"decoding":"greedy","max_new_tokens":MAX_NEW_TOKENS,"inference_seed":INFERENCE_SEED,"dataset":"SL7_v16_frozen"}
(OUT/"environment.json").write_text(json.dumps(env,indent=2)); print(json.dumps(env,indent=2))


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

{
  "model_key": "qwen3_vl_2b",
  "model_display": "Qwen3-VL-2B-Instruct",
  "model_id": "Qwen/Qwen3-VL-2B-Instruct",
  "parameters": 2127532032,
  "transformers": "5.17.0",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "python": "3.13.15",
  "dtype": "float16",
  "attention": "sdpa",
  "decoding": "greedy",
  "max_new_tokens": 64,
  "inference_seed": 20261004,
  "dataset": "SL7_v16_frozen"
}


In [5]:
def run_item(it):
    content=[]
    for rel in [p for p in it["image_paths"].split("|") if p]:
        content.append({"type":"image","path":str(ROOT/rel)})
    content.append({"type":"text","text":it["prompt"]})
    messages=[{"role":"user","content":content}]
    t0=time.time()
    inputs=processor.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt")
    # BatchFeature.to preserves integer tensors and casts floating tensors.
    try:
        inputs=inputs.to(model.device,dtype=torch.float16)
    except TypeError:
        inputs=inputs.to(model.device)
        for k,v in list(inputs.items()):
            if torch.is_tensor(v) and torch.is_floating_point(v): inputs[k]=v.to(torch.float16)
    inputs.pop("token_type_ids",None)
    with torch.inference_mode():
        generated=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False)
    new=generated[:,inputs["input_ids"].shape[1]:]
    reply=processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()
    return {"item_id":it["item_id"],"raw_output":reply,"seconds":round(time.time()-t0,2)}


## Family-specific calibration mapping

| SL-7 family | Required calibration skills |
|---|---|
| B1 Zig-zag direction reversal | number reading |
| B2 Hidden reconstruction | number reading |
| B3 Endpoint/absence anchoring | endpoint recognition |
| B4 Perception→action transport | number + endpoint |
| B5 Multi-image positional shortcut | number + endpoint + multi-image |
| B6 Attribute-conditioned counting | number + endpoint |
| B7 Multi-source visual composition | number + die + multi-image |

This mapping is used only to decide whether a model/family cell can enter the **headline cross-model confirmation matrix**. It does not change any model answer or family success threshold.


In [6]:
RAW=OUT/"raw_outputs.csv"
outrows=[]

def save_raw():
    with open(RAW,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["item_id","raw_output","seconds"])
        w.writeheader(); w.writerows(outrows)

# Frozen calibration is always run first.
for idx,it in enumerate(CAL,1):
    outrows.append(run_item(it)); save_raw(); print(f"CAL {idx}/{len(CAL)}",it["item_id"],"=>",outrows[-1]["raw_output"][:160])

pred={r["item_id"]:r["raw_output"] for r in outrows}
groups={}
for it in CAL:
    parsed=score.parse_strict(pred[it["item_id"]],it["answer_format"])
    ok=parsed.lower()==it["correct_answer"].lower() if parsed else False
    groups.setdefault(it["task_family"],[]).append(ok)
cal_counts={k:(sum(v),len(v)) for k,v in groups.items()}
requirements={"CAL_number":7,"CAL_endpoint":3,"CAL_die":4,"CAL_multi_image":3}
cal_pass={k:(sum(groups.get(k,[]))>=need) for k,need in requirements.items()}
failed=[k for k,v in cal_pass.items() if not v]

# Family-specific prerequisite mapping for the cross-model extension.
# This is deliberately conservative: a family is headline-eligible only when
# every calibration skill it directly depends on has passed.
family_gate_requirements={
    "B1_zigzag_direction":["CAL_number"],
    "B2_hidden_reconstruction":["CAL_number"],
    "B3_endpoint_anchoring":["CAL_endpoint"],
    "B4_perception_to_action":["CAL_number","CAL_endpoint"],
    "B5_multi_image_position":["CAL_number","CAL_endpoint","CAL_multi_image"],
    "B6_conditioned_counting":["CAL_number","CAL_endpoint"],
    "B7_multi_source_composition":["CAL_number","CAL_die","CAL_multi_image"],
}
family_eligibility={fam:all(cal_pass[g] for g in gates) for fam,gates in family_gate_requirements.items()}

cal_record={
    "counts":{k:list(v) for k,v in cal_counts.items()},
    "requirements":requirements,
    "calibration_pass":cal_pass,
    "original_global_gate_passed":not failed,
    "failed_calibrations":failed,
    "cross_model_protocol":"family_specific_calibration_gating_v1",
    "family_gate_requirements":family_gate_requirements,
    "family_eligibility":family_eligibility,
}
(OUT/"calibration_status.json").write_text(json.dumps(cal_record,indent=2))

print("Calibration:",cal_counts)
print("Calibration pass map:",cal_pass)
print("\nFamily eligibility for headline blind-spot confirmation:")
for fam,ok in family_eligibility.items():
    print(" ",fam,":","ELIGIBLE" if ok else "INELIGIBLE / diagnostic only", family_gate_requirements[fam])

if failed:
    print("\nOriginal global gate would FAIL for:",failed)
    print("FAMILY-GATED MODE: continuing to the 150-item frozen holdout.")
    print("Failed calibration families only make their dependent blind-spot families INELIGIBLE for headline confirmation.")
else:
    print("\nAll calibration gates passed; all seven families are eligible.")


CAL 1/20 CAL_NUM_01 => 7
CAL 2/20 CAL_NUM_02 => 15
CAL 3/20 CAL_NUM_03 => 24
CAL 4/20 CAL_NUM_04 => 33
CAL 5/20 CAL_NUM_05 => 45
CAL 6/20 CAL_NUM_06 => 58
CAL 7/20 CAL_NUM_07 => 74
CAL 8/20 CAL_NUM_08 => 97
CAL 9/20 CAL_END_01 => snake head
CAL 10/20 CAL_END_02 => snake tail
CAL 11/20 CAL_END_03 => ladder foot
CAL 12/20 CAL_END_04 => ladder top
CAL 13/20 CAL_DIE_01 => 1
CAL 14/20 CAL_DIE_02 => 4
CAL 15/20 CAL_DIE_03 => 6
CAL 16/20 CAL_DIE_04 => 3
CAL 17/20 CAL_CMP_01 => B
CAL 18/20 CAL_CMP_02 => B
CAL 19/20 CAL_CMP_03 => B
CAL 20/20 CAL_CMP_04 => B
Calibration: {'CAL_number': (8, 8), 'CAL_endpoint': (4, 4), 'CAL_die': (4, 4), 'CAL_multi_image': (2, 4)}
Calibration pass map: {'CAL_number': True, 'CAL_endpoint': True, 'CAL_die': True, 'CAL_multi_image': False}

Family eligibility for headline blind-spot confirmation:
  B1_zigzag_direction : ELIGIBLE ['CAL_number']
  B2_hidden_reconstruction : ELIGIBLE ['CAL_number']
  B3_endpoint_anchoring : ELIGIBLE ['CAL_endpoint']
  B4_perception_to_a

In [7]:
# Run the unchanged 150-item frozen holdout. We retain raw outputs even for
# calibration-ineligible families, but those families are diagnostic only.
for idx,it in enumerate(SCORED,1):
    outrows.append(run_item(it)); save_raw()
    if idx%10==0 or idx==len(SCORED): print(f"SCORED {idx}/{len(SCORED)} complete")
print("Raw outputs saved:",RAW)


SCORED 10/150 complete
SCORED 20/150 complete
SCORED 30/150 complete
SCORED 40/150 complete
SCORED 50/150 complete
SCORED 60/150 complete
SCORED 70/150 complete
SCORED 80/150 complete
SCORED 90/150 complete
SCORED 100/150 complete
SCORED 110/150 complete
SCORED 120/150 complete
SCORED 130/150 complete
SCORED 140/150 complete
SCORED 150/150 complete
Raw outputs saved: /kaggle/working/results_SL7_qwen3_vl_2b/raw_outputs.csv


In [8]:
scored_rows,family_rows,original_cal_pass=score.score(ROOT/"dataset/items.csv",RAW,OUT)

# Add family-specific calibration eligibility to the scorer output without
# changing any frozen family scoring/confirmation threshold.
fam_csv=OUT/"family_summary.csv"
fam_rows=list(csv.DictReader(open(fam_csv,encoding="utf-8")))
for r in fam_rows:
    fam=r["family"]
    eligible=bool(family_eligibility.get(fam,False))
    scorer_confirmed=(r["confirmed"]=="1")
    r["calibration_eligible"]="1" if eligible else "0"
    r["headline_confirmed"]="1" if (eligible and scorer_confirmed) else "0"
    reqs=family_gate_requirements.get(fam,[])
    failed_reqs=[g for g in reqs if not cal_pass.get(g,False)]
    r["eligibility_note"]="eligible" if eligible else "failed calibration: "+",".join(failed_reqs)

fg_csv=OUT/"family_summary_family_gated.csv"
fields=list(fam_rows[0].keys())
with open(fg_csv,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(fam_rows)

# Paper-facing summary.
L=["# SL-7 Cross-Model Family-Gated Summary","",
   f"Model: **{MODEL_DISPLAY}**",
   "",
   "The frozen v16 stimuli and family confirmation criteria are unchanged. The cross-model extension uses family-specific calibration eligibility so an unrelated failed calibration does not invalidate otherwise interpretable families.",
   "", "## Calibration", "",
   "| Calibration | Correct | Required | Pass |","|---|---:|---:|---|"]
for name in ["CAL_number","CAL_endpoint","CAL_die","CAL_multi_image"]:
    k,n=cal_counts.get(name,(0,0)); need=requirements[name]
    L.append(f"| {name.replace('CAL_','')} | {k}/{n} | {need}/{n} | {'PASS' if cal_pass[name] else 'FAIL'} |")
L += ["", "## Family-level headline eligibility and results", "",
      "| Family | Eligible? | Frozen scorer confirmed? | Headline confirmed? | Control/component | Probe | Gap | Note |",
      "|---|---|---|---|---:|---:|---:|---|"]
for r in fam_rows:
    elig="YES" if r["calibration_eligible"]=="1" else "NO"
    conf="YES" if r["confirmed"]=="1" else "NO"
    head="YES" if r["headline_confirmed"]=="1" else "NO"
    L.append(f"| {r['family']} | {elig} | {conf} | **{head}** | {r['control_or_component_correct']}/{r['control_or_component_n']} | {r['probe_correct']}/{r['probe_n']} | {r['gap_percentage_points']} pp | {r['eligibility_note']} |")
L += ["",
      "### Interpretation",
      "",
      "- `headline_confirmed=1` requires **both** the unchanged frozen family criterion and the relevant calibration prerequisites.",
      "- Results from calibration-ineligible families are retained only as diagnostics and must not be described as confirmed blind spots.",
      "- This cross-model gating amendment was made before examining Qwen3-VL-2B scored holdout outputs; no scored thresholds or stimuli were changed.",
      ""]
summary_fg=OUT/"summary_family_gated.md"
summary_fg.write_text("\n".join(L),encoding="utf-8")
print(summary_fg.read_text())

zip_path=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}_family_gated.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.name)
print("Download:",zip_path)


# SL-7 Cross-Model Family-Gated Summary

Model: **Qwen3-VL-2B-Instruct**

The frozen v16 stimuli and family confirmation criteria are unchanged. The cross-model extension uses family-specific calibration eligibility so an unrelated failed calibration does not invalidate otherwise interpretable families.

## Calibration

| Calibration | Correct | Required | Pass |
|---|---:|---:|---|
| number | 8/8 | 7/8 | PASS |
| endpoint | 4/4 | 3/4 | PASS |
| die | 4/4 | 4/4 | PASS |
| multi_image | 2/4 | 3/4 | FAIL |

## Family-level headline eligibility and results

| Family | Eligible? | Frozen scorer confirmed? | Headline confirmed? | Control/component | Probe | Gap | Note |
|---|---|---|---|---:|---:|---:|---|
| B1_zigzag_direction | YES | YES | **YES** | 10/10 | 1/10 | 90.0 pp | eligible |
| B2_hidden_reconstruction | YES | YES | **YES** | 10/10 | 0/10 | 100.0 pp | eligible |
| B3_endpoint_anchoring | YES | NO | **NO** | 6/10 | 1/10 | 50.0 pp | eligible |
| B4_perception_to_action | YES | NO |